# 4. Phylogeny

### Notebook Structure

**1.** De novo tree construction  
&nbsp;&nbsp;&nbsp;&nbsp;**1.1** Sequence Alignment   
&nbsp;&nbsp;&nbsp;&nbsp;**1.2** Alignment Masking   
&nbsp;&nbsp;&nbsp;&nbsp;**1.3** Tree Construction  
&nbsp;&nbsp;&nbsp;&nbsp;**1.4** Tree Visualization  
&nbsp;&nbsp;&nbsp;&nbsp;**1.5** Bootstrapping  
**2.** Fragment insertion  
&nbsp;&nbsp;&nbsp;&nbsp;**2.1** Silva Reference  
&nbsp;&nbsp;&nbsp;&nbsp;**2.2** Tree Construction using SEPP  
&nbsp;&nbsp;&nbsp;&nbsp;**2.3** Tree Visualization  

Building a phylogeny from 16S RNA requires first aligning our sequences and then building the phylogeny that represents the distances between these sequences.

<div style="border: 2px solid red; padding: 10px; border-radius: 5px; color: red; font-size: 90%;">
<b>Note:</b> We have a processed metadata now with the additional information, but I did not dare to start changing everything in this notebook so it is still using the raw one. Should not make any difference. Do we want to change it or leave it?
</div>

In [1]:
# 1- Import packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2

%matplotlib inline

We are using empress for the tree visualization. Since it is not part of qiime, please use "pip install empress" in the terminal before running the cells below.

In [2]:
import empress

/opt/conda/lib/python3.10/site-packages/empress/core.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# 2 - Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/Project/MicrobiomeAnalysis_TummyTribe/scripts").


In [4]:
# 3 - Data directories
meta_data_dir = "../data/processed/metadata"
raw_data_dir = "../data/raw"
phylogeny_data_dir = "../data/processed/phylogeny"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"

In [5]:
%%bash -s "$phylogeny_data_dir"
mkdir -p "$1"

## 1. De novo tree construction
We can create a phylogenetic tree by aligning the marker genes across divergent taxa and try to reconstruct the tree based on the resulting alignment. One of the issues of this approach is that short sequences may not carry enough information to capture a meaningful phylogeny. But if we include bootstrapping to ensure the robustness of our tree we shouldn't have to worry about that.
Fragment insertion is described as an alternative method further down in this notebook.

### 1.1 Sequence alignemnt

In [10]:
! qiime alignment mafft \
    --i-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --o-alignment $phylogeny_data_dir/aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/aligned-rep-seqs.qza


### 1.2 Alignment masking

In [11]:
! qiime alignment mask \
    --i-alignment $phylogeny_data_dir/aligned-rep-seqs.qza \
    --o-masked-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/masked-aligned-rep-seqs.qza


### 1.3 Tree construction

In [12]:
! qiime phylogeny fasttree \
    --i-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza \
    --o-tree $phylogeny_data_dir/fasttree-tree.qza

! qiime phylogeny midpoint-root \
    --i-tree $phylogeny_data_dir/fasttree-tree.qza \
    --o-rooted-tree $phylogeny_data_dir/fasttree-tree-rooted.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Unrooted] to: ../data/processed/phylogeny/fasttree-tree.qza
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Rooted] to: ../data/processed/phylogeny/fasttree-tree-rooted.qza


### 1.4 Tree visualization

In [8]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $phylogeny_data_dir/fasttree-tree-rooted.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/fasttree-tree-rooted.qzv


In [ ]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $metadata_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/fasttree-community-tree-viz.qzv

In [ ]:
Visualization.load(f"{phylogeny_data_dir}/fasttree-community-tree-viz.qzv")

### 1.5 Bootstrapping
The simplest test of phylogenetic accuracy is the bootstrap. With bootstrapping we essentially test whether our whole dataset is supporting our tree. This is done by taking random subsamples of the dataset, building trees from each of these and calculating the frequency with which the various parts of our de novo tree are reproduced in each of these random subsamples [Baldauf SL. Phylogeny for the faint of heart: a tutorial. Trends Genet. 2003 Jun;19(6):345-51. doi: 10.1016/S0168-9525(03)00112-4.].

This cell does run in the notebook, but you might want to get a coffee (or two) since it takes about 4 hours. 

In [4]:
! qiime phylogeny raxml-rapid-bootstrap \
    --i-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza \
    --p-seed 1723 \
    --p-rapid-bootstrap-seed 9384 \
    --p-bootstrap-replicates 100 \
    --p-substitution-model GTRCAT \
    --p-n-threads 3 \
    --o-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree.qza

IndentationError: unexpected indent (438486707.py, line 2)

In [6]:
! qiime phylogeny midpoint-root \
    --i-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree.qza \
    --o-rooted-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree-rooted.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Rooted] to: ../data/processed/phylogeny/raxml-cat-bootstrap-tree-rooted.qza


In [8]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree-rooted.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $metadata_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/raxml-cat-bootstrap-tree.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/raxml-cat-bootstrap-tree.qzv


In [9]:
Visualization.load(f"{phylogeny_data_dir}/raxml-cat-bootstrap-tree.qzv")

<visualization: Visualization uuid: d18b77c0-007e-4b6f-9bc3-1b0540f164a5>

Comparing the original de novo tree with the tree after bootstrapping, they look very similar, which is what we would expect. We will use the bootstrap tree for any downstream analysis.

## 2. Fragment insertion

Fragment insertion uses a reference tree and then our sequences are inserted into this tree. This results in a way larger tree than if we construct a de novo tree since it includes the entire reference backbone tree. We can get the SILVA 128 reference tree from qiime. According to a user in the qiime forum using SILVA 138 for the taxonomy and SILVA 128 to build the phylogenetic tree should not lead to any complications (https://forum.qiime2.org/t/compatibility-of-sepp-silva128-with-taxonomy-classification-using-silva138/31172?utm_source=chatgpt.com).

For our analysis we have decided to use the de novo tree since it is easier to construct and works well with the dataset that we are using.

### 2.1 Silva Reference 

In [18]:
#SILVA 128 for SEPP
! wget -O $phylogeny_data_dir/silva-128-sepp-refs.qza https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza

--2025-11-11 13:28:36--  https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza [following]
--2025-11-11 13:28:36--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.246.176, 52.92.162.192, 52.218.236.160, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.218.246.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 181253322 (173M) [binary/octet-stream]
Saving to: ‘../data/processed/phylogeny/silva-128-sepp-refs.qza’

../data/processed/p 100%[===================>] 172.86M  16.3MB/s    in 12s     

2025-11-11

### 2.2 Tree construction using SEPP
The following cell does not run in this notebook due to limited memory. It was run on euler and the finished tree and tree placements can be imported for visualization and further analysis.

In [21]:
#does not run in notebook -> euler (needs amplicon env)
! qiime fragment-insertion sepp \
    --i-representative-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --i-reference-database $phylogeny_data_dir/silva-128-sepp-refs.qza \
    --p-threads 2 \
    --o-tree $phylogeny_data_dir/sepp-tree.qza \
    --o-placements $phylogeny_data_dir/sepp-tree-placements.qza \
    --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Removing /tmp/tmp.1ZZbpa3ZS5/sepp-tmp-QfUBleakIt


Check if outpoot tree has expected format. It's already rooted and has Newick tree format.

### 2.3 Tree visualization

In [9]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $phylogeny_data_dir/sepp-tree.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/sepp-tree.qzv


In [16]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $metadata_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/sepp-community-tree-viz-filtered.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/sepp-community-tree-viz-filtered.qzv


In [17]:
Visualization.load(f"{phylogeny_data_dir}/sepp-community-tree-viz-filtered.qzv")

<visualization: Visualization uuid: 048bd4e2-0461-4045-b58e-e03f60de5ca8>